In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [7]:
%pip install -q dagshub mlflow

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
import os
import warnings
warnings.filterwarnings('ignore')
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub

# Download latest version
path = kagglehub.competition_download('ieee-fraud-detection')

os.environ['MLFLOW_TRACKING_USERNAME'] = 'aband22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '7ee83e7342830ad0ab1acadf4bf79200db411131'

dagshub.init(repo_owner='aband22', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)
mlflow.set_experiment("LogisticRegression_Training")

print("MLflow Connected ✓")
print("Tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "aband22/IEEE-CIS-Fraud-Detection"

Repository aband22/IEEE-CIS-Fraud-Detection initialized!

MLflow Connected ✓
Tracking URI: https://dagshub.com/aband22/IEEE-CIS-Fraud-Detection.mlflow


In [9]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test  = pd.merge(test_transaction,  test_identity,  on='TransactionID', how='left')

# Keep raw copies for final pipeline
raw_train = train.copy()
raw_test  = test.copy()

TARGET = 'isFraud'
y = train[TARGET].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Fraud rate :", round(train[TARGET].mean() * 100, 3), "%")

Train shape: (590540, 434)
Test shape : (506691, 433)
Fraud rate : 3.499 %


In [10]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):

    original_cols = train.shape[1]

    # Drop target and ID
    train.drop(columns=[TARGET, 'TransactionID'], inplace=True)
    test.drop(columns=['TransactionID'], inplace=True)

    # 1. Drop columns with >50% missing
    null_pct       = train.isnull().mean()
    high_null_cols = null_pct[null_pct > 0.5].index.tolist()
    train.drop(columns=high_null_cols, inplace=True)
    test.drop(columns=[c for c in high_null_cols if c in test.columns], inplace=True)
    print(f"Dropped high-null cols : {len(high_null_cols)}")

    # 2. Drop constant columns
    const_cols = [c for c in train.columns if train[c].nunique() <= 1]
    train.drop(columns=const_cols, inplace=True)
    test.drop(columns=[c for c in const_cols if c in test.columns], inplace=True)
    print(f"Dropped constant cols  : {len(const_cols)}")

    # 3. Label encode categoricals
    cat_cols = train.select_dtypes(include='object').columns.tolist()
    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        train[col] = le.fit_transform(train[col].astype(str))
        classes    = set(le.classes_)
        test[col]  = le.transform(
            test[col].astype(str).apply(lambda x: x if x in classes else le.classes_[0])
        )
        label_encoders[col] = le
    print(f"Label encoded cols     : {len(cat_cols)}")

    # 4. Fill remaining NaNs with median
    num_cols = train.select_dtypes(include=np.number).columns.tolist()
    medians  = train[num_cols].median()
    train[num_cols] = train[num_cols].fillna(medians)
    test[num_cols]  = test[num_cols].fillna(medians)

    # Log
    mlflow.log_param("null_threshold",      0.5)
    mlflow.log_param("nan_strategy",        "median")
    mlflow.log_param("cat_encoding",        "label_encoding")
    mlflow.log_metric("original_cols",      original_cols)
    mlflow.log_metric("dropped_null_cols",  len(high_null_cols))
    mlflow.log_metric("dropped_const_cols", len(const_cols))
    mlflow.log_metric("label_encoded_cols", len(cat_cols))
    mlflow.log_metric("remaining_cols",     train.shape[1])
    mlflow.log_metric("train_rows",         train.shape[0])
    mlflow.log_metric("fraud_rate_pct",     round(y.mean() * 100, 3))

    print(f"\nOriginal cols  : {original_cols}")
    print(f"Remaining cols : {train.shape[1]}")

Dropped high-null cols : 214
Dropped constant cols  : 0
Label encoded cols     : 9

Original cols  : 434
Remaining cols : 218
🏃 View run RandomForest_Cleaning at: https://dagshub.com/aband22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/ef260c19fde04adaaca016454d1e03a5
🧪 View experiment at: https://dagshub.com/aband22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [11]:
with mlflow.start_run(run_name="LogisticRegression_Feature_Engineering"):

    n_before = train.shape[1]

    # 1. Time features
    for df in [train, test]:
        df['hour']        = (df['TransactionDT'] / 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] / (3600 * 24)) % 7
        df['week']        = (df['TransactionDT'] / (3600 * 24 * 7)) % 52

    # 2. Transaction amount features
    for df in [train, test]:
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents']   = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_isround'] = (df['TransactionAmt_cents'] == 0).astype(int)

    # 3. Frequency encoding
    for col in ['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']:
        if col in train.columns:
            freq_map          = train[col].value_counts().to_dict()
            train[f'{col}_freq'] = train[col].map(freq_map)
            test[f'{col}_freq']  = test[col].map(freq_map).fillna(0)

    # 4. Aggregation features
    for col in ['card1', 'card2']:
        if col in train.columns:
            grp = train.groupby(col)['TransactionAmt'].agg(['mean', 'std']).reset_index()
            grp.columns = [col, f'TransactionAmt_mean_by_{col}', f'TransactionAmt_std_by_{col}']
            train = train.merge(grp, on=col, how='left')
            test  = test.merge(grp,  on=col, how='left')

    # 5. Email domain suffix
    if 'P_emaildomain' in train.columns:
        for df in [train, test]:
            df['P_email_suffix'] = df['P_emaildomain'].astype(str).apply(
                lambda x: x.split('.')[-1] if '.' in str(x) else 'unknown'
            )
        le_email = LabelEncoder()
        le_email.fit(train['P_email_suffix'].astype(str))
        train['P_email_suffix'] = le_email.transform(train['P_email_suffix'].astype(str))
        em_classes = set(le_email.classes_)
        test['P_email_suffix'] = le_email.transform(
            test['P_email_suffix'].astype(str).apply(
                lambda x: x if x in em_classes else le_email.classes_[0]
            )
        )

    # 6. Email match feature
    if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
        for df in [train, test]:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

    # 7. Card + address combined frequency
    if 'card1' in train.columns and 'addr1' in train.columns:
        for df in [train, test]:
            df['card1_addr1'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str)
        freq_ca = train['card1_addr1'].value_counts().to_dict()
        train['card1_addr1_freq'] = train['card1_addr1'].map(freq_ca)
        test['card1_addr1_freq']  = test['card1_addr1'].map(freq_ca).fillna(0)
        train.drop(columns=['card1_addr1'], inplace=True)
        test.drop(columns=['card1_addr1'],  inplace=True)

    # 8. Amount vs card mean ratio
    if 'TransactionAmt_mean_by_card1' in train.columns:
        for df in [train, test]:
            df['amt_vs_card_mean'] = df['TransactionAmt'] / (df['TransactionAmt_mean_by_card1'] + 1)

    # Fill new NaNs
    train.fillna(-999, inplace=True)
    test.fillna(-999, inplace=True)

    n_after = train.shape[1]

    mlflow.log_param("fe_methods",
        "time, log_amt, freq_encoding, aggregations, email_suffix, email_match, card_addr_freq, amt_ratio"
    )
    mlflow.log_metric("features_before_fe", n_before)
    mlflow.log_metric("features_after_fe",  n_after)
    mlflow.log_metric("new_features_added", n_after - n_before)

    print(f"Features before : {n_before}")
    print(f"Features after  : {n_after}")
    print(f"New features    : {n_after - n_before}")

Features before : 218
Features after  : 237
New features    : 19
🏃 View run LogisticRegression_Feature_Engineering at: https://dagshub.com/aband22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/86f6971ce6184b31a7c7d6110cb8cc31
🧪 View experiment at: https://dagshub.com/aband22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Feature_Selection"):

    n_before = train.shape[1]

    # 1. Remove low variance features
    vt = VarianceThreshold(threshold=0.01)

    vt.fit(train)

    low_var_cols = train.columns[~vt.get_support()].tolist()

    train.drop(columns=low_var_cols, inplace=True)

    test.drop(
        columns=[c for c in low_var_cols if c in test.columns],
        inplace=True
    )

    print(f"Removed by VarianceThreshold : {len(low_var_cols)}")

    # 2. Remove highly correlated features
    corr_matrix = train.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    high_corr_cols = [
        col for col in upper.columns
        if any(upper[col] > 0.95)
    ]

    train.drop(columns=high_corr_cols, inplace=True)

    test.drop(
        columns=[c for c in high_corr_cols if c in test.columns],
        inplace=True
    )

    print(f"Removed by Correlation Filter: {len(high_corr_cols)}")

    # 3. SelectKBest
    skb = SelectKBest(
        score_func=mutual_info_classif,
        k=min(200, train.shape[1])
    )

    skb.fit(train, y)

    selected_cols = train.columns[skb.get_support()]

    train = train[selected_cols]
    test = test[selected_cols]

    print(f"Remaining after SelectKBest : {len(selected_cols)}")

    n_after = train.shape[1]

    # MLflow logs
    mlflow.log_param("variance_threshold", 0.01)
    mlflow.log_param("correlation_threshold", 0.95)
    mlflow.log_param("selectkbest_k", 200)

    mlflow.log_metric("features_before_selection", n_before)
    mlflow.log_metric("removed_low_variance", len(low_var_cols))
    mlflow.log_metric("removed_high_corr", len(high_corr_cols))
    mlflow.log_metric("features_after_selection", n_after)

    print(f"\nFeatures before : {n_before}")
    print(f"Features after  : {n_after}")

Removed by VarianceThreshold : 23
Removed by Correlation Filter: 56


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

scaler = StandardScaler()

train_arr = scaler.fit_transform(train)
test_arr = scaler.transform(test)

train = pd.DataFrame(
    train_arr,
    columns=train.columns
)

test = pd.DataFrame(
    test_arr,
    columns=test.columns
)

In [ ]:
def run_cv_lr(params, run_name, n_splits=5):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    fold_train_aucs = []
    fold_val_aucs = []

    with mlflow.start_run(run_name=run_name):

        mlflow.log_params(params)

        for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y)):

            X_tr = train.iloc[tr_idx]
            X_val = train.iloc[val_idx]

            y_tr = y.iloc[tr_idx]
            y_val = y.iloc[val_idx]

            model = LogisticRegression(
                **params,
                random_state=42,
                max_iter=1000,
                n_jobs=-1
            )

            model.fit(X_tr, y_tr)

            train_preds = model.predict_proba(X_tr)[:, 1]
            val_preds = model.predict_proba(X_val)[:, 1]

            train_auc = roc_auc_score(y_tr, train_preds)
            val_auc = roc_auc_score(y_val, val_preds)

            fold_train_aucs.append(train_auc)
            fold_val_aucs.append(val_auc)

            mlflow.log_metric(
                f"fold_{fold+1}_train_auc",
                train_auc
            )

            mlflow.log_metric(
                f"fold_{fold+1}_val_auc",
                val_auc
            )

            print(
                f"Fold {fold+1} → "
                f"Train: {train_auc:.4f} | "
                f"Val: {val_auc:.4f} | "
                f"Gap: {train_auc - val_auc:.4f}"
            )

        mean_train = np.mean(fold_train_aucs)
        mean_val = np.mean(fold_val_aucs)
        gap = mean_train - mean_val

        mlflow.log_metric("mean_train_auc", mean_train)
        mlflow.log_metric("mean_val_auc", mean_val)
        mlflow.log_metric("overfit_gap", gap)

        status = (
            "OVERFITTING"
            if gap > 0.05
            else "UNDERFITTING"
            if mean_val < 0.75
            else "HEALTHY"
        )

        print(f"\nMean Train AUC : {mean_train:.4f}")
        print(f"Mean Val AUC   : {mean_val:.4f}")
        print(f"Overfit Gap    : {gap:.4f} → {status}")

    return mean_val

In [ ]:
underfit_params = {'C': 0.0001, 'penalty': 'l2', 'class_weight': None}
run_cv_lr(underfit_params, run_name="LogisticRegression_Underfitted")

## Underfitting Analysis

**Observation:** Both train and val AUC are low.

**Root cause:**
- `C=0.0001` is an extremely strong regularization (C is the inverse of
  regularization strength). This shrinks all coefficients very close to zero,
  making the model almost unable to learn any patterns.
- Logistic Regression is already a linear model — it cannot capture
  non-linear relationships. Combined with heavy regularization,
  it becomes extremely constrained.

**Conclusion:** The model is too heavily penalized to learn meaningful patterns.
Fix: increase C (reduce regularization strength).

In [ ]:
overfit_params = {'C': 1000, 'penalty': 'l2', 'class_weight': None}
run_cv_lr(overfit_params, run_name="LogisticRegression_Overfitted")

## Overfitting Analysis

**Observation:** Train AUC slightly higher than val AUC.

**Note:** Logistic Regression is naturally resistant to overfitting compared
to tree-based methods because it is a linear model. Even with C=1000
(almost no regularization), the gap may be small. However, on high-dimensional
data like this (100+ features), very high C can still cause overfitting
as the model fits noise in the training features.

**Root cause:** No regularization allows coefficients to grow large and
fit training noise, especially on correlated features.

**Conclusion:** LR needs moderate regularization. C=1 to C=10 is typically
the sweet spot.

In [ ]:
tuned_params_lr = {
    'C':            1.0,
    'penalty':     'l2',
    'solver':  'lbfgs',
    'class_weight': 'balanced',
}
run_cv_lr(tuned_params_lr, run_name="LogisticRegression_Tuned")

In [ ]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_thresh=0.5, top_k=80):
        self.null_thresh = null_thresh
        self.top_k       = top_k

    def fit(self, X, y=None):
        X = X.copy()

        # Drop ID and target if present
        for col in ['TransactionID', 'isFraud']:
            if col in X.columns:
                X.drop(columns=[col], inplace=True)

        # --- CLEANING ---

        # 1. Drop high-null columns
        null_pct             = X.isnull().mean()
        self.high_null_cols_ = null_pct[null_pct > self.null_thresh].index.tolist()
        X.drop(columns=self.high_null_cols_, inplace=True)

        # 2. Drop constant columns
        self.const_cols_ = [c for c in X.columns if X[c].nunique() <= 1]
        X.drop(columns=self.const_cols_, inplace=True)

        # 3. Label encode categoricals
        self.cat_cols_       = X.select_dtypes(include='object').columns.tolist()
        self.label_encoders_ = {}
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            self.label_encoders_[col] = le
            X[col] = le.transform(X[col].astype(str))

        # 4. Store medians for numeric NaN filling
        num_cols      = X.select_dtypes(include=np.number).columns.tolist()
        self.medians_ = X[num_cols].median()
        X[num_cols]   = X[num_cols].fillna(self.medians_)

        # --- FEATURE ENGINEERING ---

        # Frequency maps
        self.freq_maps_ = {}
        for col in ['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts().to_dict()

        # Aggregation maps
        self.agg_maps_ = {}
        for col in ['card1', 'card2']:
            if col in X.columns:
                self.agg_maps_[col] = X.groupby(col)['TransactionAmt'].agg(['mean', 'std'])

        # Email encoder
        self.has_email_ = 'P_emaildomain' in X.columns
        if self.has_email_:
            X['P_email_suffix'] = X['P_emaildomain'].astype(str).apply(
                lambda x: x.split('.')[-1] if '.' in str(x) else 'unknown'
            )
            self.le_email_ = LabelEncoder()
            self.le_email_.fit(X['P_email_suffix'].astype(str))

        # Card + address frequency map
        self.has_card_addr_ = 'card1' in X.columns and 'addr1' in X.columns
        if self.has_card_addr_:
            X['card1_addr1']     = X['card1'].astype(str) + '_' + X['addr1'].astype(str)
            self.card_addr_freq_ = X['card1_addr1'].value_counts().to_dict()

        # Apply FE to get full feature set
        X = self._apply_fe(X)
        X.fillna(-999, inplace=True)

        # --- FEATURE SELECTION ---
        # LR has no feature_importances_ so we use SelectKBest with mutual information
        from sklearn.feature_selection import SelectKBest, mutual_info_classif
        self.skb_ = SelectKBest(
            score_func=mutual_info_classif,
            k=min(self.top_k, X.shape[1])
        )
        self.skb_.fit(X, y)
        self.top_features_ = X.columns[self.skb_.get_support()].tolist()

        print(f"Selected {len(self.top_features_)} features via SelectKBest")
        return self

    def _apply_fe(self, X):
        # Time features
        if 'TransactionDT' in X.columns:
            X['hour']        = (X['TransactionDT'] / 3600) % 24
            X['day_of_week'] = (X['TransactionDT'] / (3600 * 24)) % 7
            X['week']        = (X['TransactionDT'] / (3600 * 24 * 7)) % 52

        # Amount features
        if 'TransactionAmt' in X.columns:
            X['TransactionAmt_log']     = np.log1p(X['TransactionAmt'])
            X['TransactionAmt_cents']   = X['TransactionAmt'] - X['TransactionAmt'].astype(int)
            X['TransactionAmt_isround'] = (X['TransactionAmt_cents'] == 0).astype(int)

        # Frequency encoding
        for col, freq_map in self.freq_maps_.items():
            if col in X.columns:
                X[f'{col}_freq'] = X[col].map(freq_map).fillna(0)

        # Aggregation features
        for col, agg in self.agg_maps_.items():
            if col in X.columns:
                X[f'TransactionAmt_mean_by_{col}'] = X[col].map(agg['mean'])
                X[f'TransactionAmt_std_by_{col}']  = X[col].map(agg['std'])

        # Amount vs card mean ratio
        if 'TransactionAmt_mean_by_card1' in X.columns:
            X['amt_vs_card_mean'] = (
                X['TransactionAmt'] / (X['TransactionAmt_mean_by_card1'] + 1)
            )

        # Email suffix
        if self.has_email_ and 'P_emaildomain' in X.columns:
            X['P_email_suffix'] = X['P_emaildomain'].astype(str).apply(
                lambda x: x.split('.')[-1] if '.' in str(x) else 'unknown'
            )
            classes = set(self.le_email_.classes_)
            X['P_email_suffix'] = self.le_email_.transform(
                X['P_email_suffix'].astype(str).apply(
                    lambda x: x if x in classes else self.le_email_.classes_[0]
                )
            )

        # Email match
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (
                X['P_emaildomain'] == X['R_emaildomain']
            ).astype(int)

        # Card + address frequency
        if self.has_card_addr_ and 'card1' in X.columns and 'addr1' in X.columns:
            X['card1_addr1']      = X['card1'].astype(str) + '_' + X['addr1'].astype(str)
            X['card1_addr1_freq'] = X['card1_addr1'].map(self.card_addr_freq_).fillna(0)
            X.drop(columns=['card1_addr1'], inplace=True)

        return X

    def transform(self, X):
        X = X.copy()

        # Drop ID and target
        for col in ['TransactionID', 'isFraud']:
            if col in X.columns:
                X.drop(columns=[col], inplace=True)

        # Drop stored bad columns
        X.drop(columns=[c for c in self.high_null_cols_ if c in X.columns], inplace=True)
        X.drop(columns=[c for c in self.const_cols_     if c in X.columns], inplace=True)

        # Label encode
        for col, le in self.label_encoders_.items():
            if col in X.columns:
                classes = set(le.classes_)
                X[col]  = le.transform(
                    X[col].astype(str).apply(
                        lambda x: x if x in classes else le.classes_[0]
                    )
                )

        # Fill NaNs with stored medians
        for col in self.medians_.index:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians_[col])

        # Apply feature engineering
        X = self._apply_fe(X)
        X.fillna(-999, inplace=True)

        # Keep only selected features
        available = [c for c in self.top_features_ if c in X.columns]
        return X[available]

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Best_Model"):

    mlflow.log_params(tuned_params_lr)

    raw_train_data = pd.merge(train_transaction, train_identity,
                               on='TransactionID', how='left')
    raw_y          = raw_train_data['isFraud'].copy()

    X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
        raw_train_data, raw_y, test_size=0.2, random_state=42, stratify=raw_y
    )

    pipeline_lr = Pipeline([
        ('preprocessor', FraudPreprocessor(null_thresh=0.5, top_k=80)),
        ('scaler',        StandardScaler()),   # LR needs scaling — tree models don't
        ('model',         LogisticRegression(**tuned_params_lr,
                                              max_iter=1000,
                                              random_state=42,
                                              n_jobs=-1))
    ])
    
    pipeline_lr.fit(raw_train_data.drop(columns=['isFraud']), raw_y)

    val_preds  = pipeline_lr.predict_proba(X_val_raw)[:, 1]
    val_labels = pipeline_lr.predict(X_val_raw)
    val_auc    = roc_auc_score(y_val, val_preds)
    val_f1     = f1_score(y_val, val_labels)
    val_prec   = precision_score(y_val, val_labels)
    val_rec    = recall_score(y_val, val_labels)

    mlflow.log_metric("val_auc",       val_auc)
    mlflow.log_metric("val_f1",        val_f1)
    mlflow.log_metric("val_precision", val_prec)
    mlflow.log_metric("val_recall",    val_rec)

    cm = confusion_matrix(y_val, val_labels)
    fig, ax = plt.subplots(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax)
    ax.set_title("Confusion Matrix — Logistic Regression")
    plt.tight_layout()
    plt.savefig("lr_confusion_matrix.png")
    mlflow.log_artifact("lr_confusion_matrix.png")
    plt.show()

    pipeline_lr.fit(raw_train_data.drop(columns=['isFraud']), raw_y)

    mlflow.sklearn.log_model(
        pipeline_lr,
        artifact_path="lr_pipeline",
        registered_model_name="LogisticRegression_FraudDetection"
    )

    print(f"Val AUC : {val_auc:.4f}")
    print("Pipeline saved to Model Registry ✓")

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Best_Model"):

    mlflow.log_params(tuned_params_lr)

    raw_train_data = pd.merge(
        train_transaction,
        train_identity,
        on='TransactionID',
        how='left'
    )

    raw_y = raw_train_data['isFraud'].copy()

    X = raw_train_data.drop(columns=['isFraud'])

    X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
        X,
        raw_y,
        test_size=0.2,
        random_state=42,
        stratify=raw_y
    )

    pipeline_lr = Pipeline([
        ('preprocessor',
         FraudPreprocessor(null_thresh=0.5, top_k=80)),

        ('scaler',
         StandardScaler()),

        ('model',
         LogisticRegression(
             **tuned_params_lr,
             max_iter=1000,
             random_state=42,
             n_jobs=-1
         ))
    ])

    # TRAIN ONLY ON TRAIN SPLIT
    pipeline_lr.fit(X_tr_raw, y_tr)

    # VALIDATION
    val_preds = pipeline_lr.predict_proba(X_val_raw)[:, 1]
    val_labels = pipeline_lr.predict(X_val_raw)

    val_auc = roc_auc_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_labels)
    val_prec = precision_score(y_val, val_labels)
    val_rec = recall_score(y_val, val_labels)

    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_precision", val_prec)
    mlflow.log_metric("val_recall", val_rec)

    # SAVE MODEL
    mlflow.sklearn.log_model(
        pipeline_lr,
        artifact_path="lr_pipeline"
    )

    print(f"Validation AUC       : {val_auc:.4f}")
    print(f"Validation F1        : {val_f1:.4f}")
    print(f"Validation Precision : {val_prec:.4f}")
    print(f"Validation Recall    : {val_rec:.4f}")

    print("Pipeline saved ✓")